In [1]:
import json
import re
import time
from datetime import datetime
from pathlib import Path

import requests
from bs4 import BeautifulSoup

In [2]:
# =========================
# 0. 기본 경로 설정
# =========================
PROJECT_ROOT = Path(r"C:\Users\asguug\Documents\rag-agent")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DOCS_DIR = PROJECT_ROOT / "data" / "docs"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"

RAW_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("DOCS_DIR:", DOCS_DIR)
print("CHROMA_DIR:", CHROMA_DIR)

PROJECT_ROOT: C:\Users\asguug\Documents\rag-agent
RAW_DIR: C:\Users\asguug\Documents\rag-agent\data\raw
DOCS_DIR: C:\Users\asguug\Documents\rag-agent\data\docs
CHROMA_DIR: C:\Users\asguug\Documents\rag-agent\data\chroma_db


In [3]:
GAMES = {
    "hollow_knight": {
        "appid": 367520,
        "title": "Hollow Knight",
    },
    "monster_hunter_world": {
        "appid": 582010,
        "title": "Monster Hunter: World",
    },
    "baldurs_gate_3": {
        "appid": 1086940,
        "title": "Baldur's Gate 3",
    },
    "no_mans_sky": {
        "appid": 275850,
        "title": "No Man's Sky",
    },
    "cyberpunk_2077": {
        "appid": 1091500,
        "title": "Cyberpunk 2077",
    },
}

GAMES

{'hollow_knight': {'appid': 367520, 'title': 'Hollow Knight'},
 'monster_hunter_world': {'appid': 582010, 'title': 'Monster Hunter: World'},
 'baldurs_gate_3': {'appid': 1086940, 'title': "Baldur's Gate 3"},
 'no_mans_sky': {'appid': 275850, 'title': "No Man's Sky"},
 'cyberpunk_2077': {'appid': 1091500, 'title': 'Cyberpunk 2077'}}

In [4]:
def clean_html(raw_html: str) -> str:
    """Steam 설명/뉴스 HTML을 일반 텍스트로 변환한다."""
    if not raw_html:
        return ""

    soup = BeautifulSoup(raw_html, "html.parser")
    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def unix_to_date(ts) -> str:
    """Unix timestamp를 YYYY-MM-DD 문자열로 변환한다."""
    if not ts:
        return ""

    try:
        return datetime.fromtimestamp(int(ts)).strftime("%Y-%m-%d")
    except Exception:
        return ""


def save_json(path: Path, data: dict) -> None:
    """원본 JSON 저장."""
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def safe_get_json(url: str, params: dict | None = None, timeout: int = 20) -> dict:
    """GET 요청 후 JSON 반환. 실패하면 빈 dict 반환."""
    try:
        response = requests.get(
            url,
            params=params,
            timeout=timeout,
            headers={
                "User-Agent": "Mozilla/5.0 (baseline-rag-study)"
            },
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"[WARN] 요청 실패: {url}")
        print(f"       이유: {e}")
        return {}

print("utils OK")

utils OK


In [5]:
def fetch_app_details(appid: int) -> dict:
    """
    Steam Store appdetails 정보 수집.
    baseline 문서 생성용으로 사용한다.
    """
    url = "https://store.steampowered.com/api/appdetails"
    params = {
        "appids": appid,
        "l": "english",
        "cc": "US",
    }
    return safe_get_json(url, params=params)


def fetch_reviews(appid: int, num_per_page: int = 20) -> dict:
    """
    Steam 최근 리뷰 수집.
    3주차 baseline에서는 20개만 사용한다.
    """
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        "json": 1,
        "filter": "recent",
        "language": "english",
        "review_type": "all",
        "purchase_type": "all",
        "num_per_page": num_per_page,
    }
    return safe_get_json(url, params=params)


def fetch_news(appid: int, count: int = 5) -> dict:
    """Steam News / Patch note 수집."""
    url = "https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/"
    params = {
        "appid": appid,
        "count": count,
        "maxlength": 2000,
        "format": "json",
    }
    return safe_get_json(url, params=params)

print("fetch functions OK")

fetch functions OK


In [6]:
def build_markdown(game_key: str, game_info: dict, app_details: dict, reviews: dict, news: dict) -> str:
    appid = game_info["appid"]
    title = game_info["title"]

    detail_data = {}
    if str(appid) in app_details:
        detail_data = app_details.get(str(appid), {}).get("data", {}) or {}

    short_description = clean_html(detail_data.get("short_description", ""))
    about_the_game = clean_html(detail_data.get("about_the_game", ""))
    detailed_description = clean_html(detail_data.get("detailed_description", ""))

    genres = detail_data.get("genres", []) or []
    genre_text = ", ".join([g.get("description", "") for g in genres if g.get("description")])

    categories = detail_data.get("categories", []) or []
    category_text = ", ".join([c.get("description", "") for c in categories if c.get("description")])

    release_date = detail_data.get("release_date", {}).get("date", "")
    developers = ", ".join(detail_data.get("developers", []) or [])
    publishers = ", ".join(detail_data.get("publishers", []) or [])

    review_items = reviews.get("reviews", []) or []
    news_items = news.get("appnews", {}).get("newsitems", []) or []

    lines = []

    lines.append(f"# {title}")
    lines.append("")
    lines.append("## Metadata")
    lines.append(f"- game_key: {game_key}")
    lines.append(f"- appid: {appid}")
    lines.append(f"- title: {title}")
    lines.append(f"- release_date: {release_date}")
    lines.append(f"- developers: {developers}")
    lines.append(f"- publishers: {publishers}")
    lines.append(f"- genres: {genre_text}")
    lines.append(f"- categories: {category_text}")
    lines.append("")

    lines.append("## Store Summary")
    lines.append(short_description if short_description else "No short description found.")
    lines.append("")

    lines.append("## About The Game")
    if about_the_game:
        lines.append(about_the_game)
    elif detailed_description:
        lines.append(detailed_description)
    else:
        lines.append("No detailed store description found.")
    lines.append("")

    lines.append("## Recent Steam Reviews")
    if review_items:
        for idx, review in enumerate(review_items, start=1):
            voted_up = review.get("voted_up")
            sentiment = "positive" if voted_up else "negative"
            created_at = unix_to_date(review.get("timestamp_created"))
            updated_at = unix_to_date(review.get("timestamp_updated"))
            playtime = review.get("author", {}).get("playtime_forever", 0)
            playtime_hours = round(playtime / 60, 1) if playtime else 0
            review_text = clean_html(review.get("review", ""))

            lines.append(f"### Review {idx}")
            lines.append(f"- sentiment: {sentiment}")
            lines.append(f"- created_at: {created_at}")
            lines.append(f"- updated_at: {updated_at}")
            lines.append(f"- playtime_hours: {playtime_hours}")
            lines.append("")
            lines.append(review_text[:1500])
            lines.append("")
    else:
        lines.append("No recent reviews found.")
        lines.append("")

    lines.append("## Steam News and Updates")
    if news_items:
        for idx, item in enumerate(news_items, start=1):
            news_title = item.get("title", "")
            news_date = unix_to_date(item.get("date"))
            contents = clean_html(item.get("contents", ""))

            lines.append(f"### News {idx}: {news_title}")
            lines.append(f"- date: {news_date}")
            lines.append("")
            lines.append(contents[:2000])
            lines.append("")
    else:
        lines.append("No news items found.")
        lines.append("")

    return "\n".join(lines)

print("markdown function OK")

markdown function OK


In [7]:
for game_key, game_info in GAMES.items():
    appid = game_info["appid"]
    title = game_info["title"]

    print(f"[INFO] Collecting: {title} ({appid})")

    app_details = fetch_app_details(appid)
    reviews = fetch_reviews(appid, num_per_page=20)
    news = fetch_news(appid, count=5)

    save_json(RAW_DIR / f"{game_key}_app_details.json", app_details)
    save_json(RAW_DIR / f"{game_key}_reviews.json", reviews)
    save_json(RAW_DIR / f"{game_key}_news.json", news)

    markdown = build_markdown(
        game_key=game_key,
        game_info=game_info,
        app_details=app_details,
        reviews=reviews,
        news=news,
    )

    output_path = DOCS_DIR / f"{game_key}.md"
    output_path.write_text(markdown, encoding="utf-8")

    print(f"[OK] Saved markdown: {output_path}")
    print("")

    time.sleep(1)

print("[DONE] Steam baseline documents created.")

[INFO] Collecting: Hollow Knight (367520)
[OK] Saved markdown: C:\Users\asguug\Documents\rag-agent\data\docs\hollow_knight.md

[INFO] Collecting: Monster Hunter: World (582010)
[OK] Saved markdown: C:\Users\asguug\Documents\rag-agent\data\docs\monster_hunter_world.md

[INFO] Collecting: Baldur's Gate 3 (1086940)
[OK] Saved markdown: C:\Users\asguug\Documents\rag-agent\data\docs\baldurs_gate_3.md

[INFO] Collecting: No Man's Sky (275850)
[OK] Saved markdown: C:\Users\asguug\Documents\rag-agent\data\docs\no_mans_sky.md

[INFO] Collecting: Cyberpunk 2077 (1091500)
[OK] Saved markdown: C:\Users\asguug\Documents\rag-agent\data\docs\cyberpunk_2077.md

[DONE] Steam baseline documents created.


In [8]:
md_files = sorted(DOCS_DIR.glob("*.md"))

print("생성된 Markdown 문서 수:", len(md_files))
for path in md_files:
    print("-", path.name)

생성된 Markdown 문서 수: 5
- baldurs_gate_3.md
- cyberpunk_2077.md
- hollow_knight.md
- monster_hunter_world.md
- no_mans_sky.md


In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader(
    path=str(DOCS_DIR),
    glob="*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

docs = loader.load()

print("로드된 문서 개수:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print("source:", doc.metadata.get("source"))
    print("content length:", len(doc.page_content))
    print(doc.page_content[:300])

로드된 문서 개수: 5

--- Document 1 ---
source: C:\Users\asguug\Documents\rag-agent\data\docs\baldurs_gate_3.md
content length: 14069
# Baldur's Gate 3

## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cros

--- Document 2 ---
source: C:\Users\asguug\Documents\rag-agent\data\docs\cyberpunk_2077.md
content length: 6752
# Cyberpunk 2077

## Metadata
- game_key: cyberpunk_2077
- appid: 1091500
- title: Cyberpunk 2077
- release_date: Dec 9, 2020
- developers: CD PROJEKT RED
- publishers: CD PROJEKT RED
- genres: RPG
- categories: Single-player, Steam Achievements, Full controller support, Steam Trading Cards, Adjusta

--- Document 3 ---
source: C:\Users\asguug\Documents\rag-agent\data\docs\hollow_knight.md
content length: 10256
# Hollow Knight

## Metadata
- game_key: holl

In [10]:
from langchain_core.documents import Document

def infer_game_key_from_source(source: str) -> str:
    filename = Path(source).stem
    return filename


def split_markdown_by_sections(doc):
    """
    하나의 게임 md 문서를 section 단위 Document로 분리한다.
    section metadata를 붙여서 review/news 검색을 구분하기 쉽게 만든다.
    """
    source = doc.metadata.get("source", "")
    game_key = infer_game_key_from_source(source)
    text = doc.page_content

    section_patterns = [
        ("metadata", "## Metadata"),
        ("store_summary", "## Store Summary"),
        ("about", "## About The Game"),
        ("review", "## Recent Steam Reviews"),
        ("news", "## Steam News and Updates"),
    ]

    sections = []

    for idx, (section_name, heading) in enumerate(section_patterns):
        start = text.find(heading)

        if start == -1:
            continue

        if idx + 1 < len(section_patterns):
            next_heading = section_patterns[idx + 1][1]
            end = text.find(next_heading, start + len(heading))
            if end == -1:
                end = len(text)
        else:
            end = len(text)

        section_text = text[start:end].strip()

        if section_text:
            sections.append(
                Document(
                    page_content=section_text,
                    metadata={
                        "source": source,
                        "game_key": game_key,
                        "section": section_name,
                    },
                )
            )

    return sections


section_docs = []

for doc in docs:
    section_docs.extend(split_markdown_by_sections(doc))

print("원본 문서 개수:", len(docs))
print("section 문서 개수:", len(section_docs))

for d in section_docs[:10]:
    print(Path(d.metadata["source"]).name, d.metadata["game_key"], d.metadata["section"], len(d.page_content))

원본 문서 개수: 5
section 문서 개수: 25
baldurs_gate_3.md baldurs_gate_3 metadata 636
baldurs_gate_3.md baldurs_gate_3 store_summary 224
baldurs_gate_3.md baldurs_gate_3 about 5376
baldurs_gate_3.md baldurs_gate_3 review 5101
baldurs_gate_3.md baldurs_gate_3 news 2704
cyberpunk_2077.md cyberpunk_2077 metadata 514
cyberpunk_2077.md cyberpunk_2077 store_summary 195
cyberpunk_2077.md cyberpunk_2077 about 65
cyberpunk_2077.md cyberpunk_2077 review 3512
cyberpunk_2077.md cyberpunk_2077 news 2439


In [11]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

splits = text_splitter.split_documents(section_docs)

print("원본 문서 개수:", len(docs))
print("section 문서 개수:", len(section_docs))
print("생성된 chunk 개수:", len(splits))
print("chunk_size:", CHUNK_SIZE)
print("chunk_overlap:", CHUNK_OVERLAP)

for i, chunk in enumerate(splits[:5], start=1):
    print("\n--- chunk", i, "---")
    print(chunk.metadata)
    print(chunk.page_content[:300])

원본 문서 개수: 5
section 문서 개수: 25
생성된 chunk 개수: 115
chunk_size: 800
chunk_overlap: 120

--- chunk 1 ---
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'section': 'metadata'}
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multipla

--- chunk 2 ---
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'section': 'store_summary'}
## Store Summary
Baldur’s Gate 3 is a story-rich, party-based RPG set in the universe of Dungeons & Dragons, where your choices shape a tale of fellowship and betrayal, survival and sacrifice, and the lure of absolute power.

--- chunk 3 ---
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\

In [12]:
for i, chunk in enumerate(splits[:3]):
    print(f"\n==================== Chunk {i+1} ====================")
    print("source:", chunk.metadata.get("source"))
    print("length:", len(chunk.page_content))
    print(chunk.page_content[:1000])


==================== Chunk 1 ====================
source: C:\Users\asguug\Documents\rag-agent\data\docs\baldurs_gate_3.md
length: 636
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, Steam Achievements, Full controller support, Steam Trading Cards, Adjustable Text Size, Camera Comfort, Color Alternatives, Custom Volume Controls, Adjustable Difficulty, Playable without Timed Input, Save Anytime, Stereo Sound, Subtitle Options, Surround Sound, Steam Cloud, Remote Play on TV, Remote Play Together, Family Sharing

==================== Chunk 2 ====================
source: C:\Users\asguug\Documents\rag-agent\data\docs\baldurs_gate_3.md
length: 224
## Store Summary
Baldur’s Gate 3 is a story-rich, party-based RPG set in the universe of Dung

In [13]:
from collections import Counter

source_counter = Counter()

for chunk in splits:
    source = Path(chunk.metadata.get("source", "")).name
    source_counter[source] += 1

for source, count in source_counter.items():
    print(f"{source}: {count} chunks")

baldurs_gate_3.md: 24 chunks
cyberpunk_2077.md: 13 chunks
hollow_knight.md: 19 chunks
monster_hunter_world.md: 30 chunks
no_mans_sky.md: 29 chunks


In [14]:
baseline_summary = {
    "document_count": len(docs),
    "chunk_count": len(splits),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "source_files": [Path(doc.metadata.get("source", "")).name for doc in docs],
}

baseline_summary

{'document_count': 5,
 'chunk_count': 115,
 'chunk_size': 800,
 'chunk_overlap': 120,
 'source_files': ['baldurs_gate_3.md',
  'cyberpunk_2077.md',
  'hollow_knight.md',
  'monster_hunter_world.md',
  'no_mans_sky.md']}

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [16]:
import shutil
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "steam_week3_baseline"

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(CHROMA_DIR),
)

print("Chroma vectorstore recreated")
print("chunk_count:", len(splits))

Chroma vectorstore recreated
chunk_count: 115


In [17]:

general_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

review_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {"section": "review"},
    },
)

news_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {"section": "news"},
    },
)

print("retrievers created")

retrievers created


In [18]:
test_cases = [
    ("general", "Hollow Knight는 어떤 플레이 스타일의 게임인가요?", general_retriever),
    ("general", "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?", general_retriever),
    ("review", "Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?", review_retriever),
    ("news", "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?", news_retriever),
]

for intent, query, selected_retriever in test_cases:
    print("\n" + "=" * 80)
    print("Intent:", intent)
    print("Query:", query)

    retrieved_docs = selected_retriever.invoke(query)

    for i, doc in enumerate(retrieved_docs, start=1):
        source = Path(doc.metadata.get("source", "")).name
        section = doc.metadata.get("section")
        preview = doc.page_content[:300].replace("\n", " ")
        print(f"[{i}] source={source} | section={section} | preview={preview}")


Intent: general
Query: Hollow Knight는 어떤 플레이 스타일의 게임인가요?
[1] source=hollow_knight.md | section=about | preview=Hollow Knight is a classically styled 2D action adventure across a vast interconnected world. Explore twisting caverns, ancient cities and deadly wastes; battle tainted creatures and befriend bizarre bugs; and solve ancient mysteries at the kingdom's heart.  Game Features Classic side-scrolling acti
[2] source=hollow_knight.md | section=store_summary | preview=## Store Summary Forge your own path in Hollow Knight! An epic action adventure through a vast ruined kingdom of insects and heroes. Explore twisting caverns, battle tainted creatures and befriend bizarre bugs, all in a classic, hand-drawn 2D style.
[3] source=hollow_knight.md | section=about | preview=Complete Hollow Knight to unlock Steel Soul Mode, the ultimate challenge!  An Evocative Hand-Crafted World The world of Hollow Knight is brought to life in vivid, moody detail, its caverns alive with bizarre and terrifyin

In [19]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env")

print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY exists: True


In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0,
)

GAME_ALIASES = {
    "hollow_knight": [
        "hollow knight", "할로우 나이트", "할로우나이트"
    ],
    "monster_hunter_world": [
        "monster hunter: world", "monster hunter world",
        "몬스터 헌터 월드", "몬헌 월드", "몬헌월드"
    ],
    "baldurs_gate_3": [
        "baldur's gate 3", "baldurs gate 3",
        "발더스 게이트 3", "발더스3", "발더스 게이트"
    ],
    "no_mans_sky": [
        "no man's sky", "no mans sky",
        "노 맨즈 스카이", "노맨즈스카이"
    ],
    "cyberpunk_2077": [
        "cyberpunk 2077", "사이버펑크 2077",
        "사펑", "사이버펑크"
    ],
}


def detect_game_key(query: str):
    query_lower = query.lower()

    for game_key, aliases in GAME_ALIASES.items():
        for alias in aliases:
            if alias.lower() in query_lower:
                return game_key

    return None


def detect_intent(query: str):
    query_lower = query.lower()

    review_keywords = [
        "review", "reviews", "recent review", "steam review",
        "리뷰", "최근 리뷰", "평가", "반응", "유저 반응", "민심"
    ]

    news_keywords = [
        "update", "updates", "patch", "news", "recent update",
        "업데이트", "패치", "뉴스", "변경", "개선"
    ]

    if any(keyword in query_lower for keyword in review_keywords):
        return "review"

    if any(keyword in query_lower for keyword in news_keywords):
        return "news"

    return "general"


def build_filter(intent: str, game_key: str | None):
    """
    intent와 game_key를 바탕으로 Chroma metadata filter 생성.
    game_key가 있으면 반드시 해당 게임 문서 안에서만 검색한다.
    """
    conditions = []

    if intent == "review":
        conditions.append({"section": {"$eq": "review"}})
    elif intent == "news":
        conditions.append({"section": {"$eq": "news"}})

    if game_key is not None:
        conditions.append({"game_key": {"$eq": game_key}})

    if len(conditions) == 0:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {"$and": conditions}


In [26]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a Steam game recommendation and analysis assistant.

Answer the user's question using only the provided context.
If the context is insufficient, say that the available document does not contain enough evidence.
Do not invent details that are not supported by the context.

Write the answer in Korean.
Keep the answer concise but grounded.
            """.strip(),
        ),
        (
            "human",
            """
[Question]
{question}

[Context]
{context}
            """.strip(),
        ),
    ]
)


def format_docs(docs):
    formatted = []

    for i, doc in enumerate(docs, start=1):
        source = Path(doc.metadata.get("source", "")).name
        section = doc.metadata.get("section", "unknown")
        game_key = doc.metadata.get("game_key", "unknown")
        content = doc.page_content

        formatted.append(
            f"[Context {i}]\n"
            f"source: {source}\n"
            f"game_key: {game_key}\n"
            f"section: {section}\n"
            f"{content}"
        )

    return "\n\n".join(formatted)


def retrieve_docs(query: str, k: int = 4):
    intent = detect_intent(query)
    game_key = detect_game_key(query)
    metadata_filter = build_filter(intent=intent, game_key=game_key)

    if metadata_filter is None:
        retrieved_docs = vectorstore.similarity_search(
            query=query,
            k=k,
        )
    else:
        retrieved_docs = vectorstore.similarity_search(
            query=query,
            k=k,
            filter=metadata_filter,
        )

    return intent, game_key, metadata_filter, retrieved_docs


def run_rag(query: str, k: int = 4):
    intent, game_key, metadata_filter, retrieved_docs = retrieve_docs(query, k=k)

    context = format_docs(retrieved_docs)

    messages = rag_prompt.format_messages(
        question=query,
        context=context,
    )

    response = llm.invoke(messages)

    return {
        "query": query,
        "intent": intent,
        "game_key": game_key,
        "metadata_filter": metadata_filter,
        "answer": response.content,
        "retrieved_docs": retrieved_docs,
        "context": context,
    }


In [27]:
result = run_rag("Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?")

print("Intent:", result["intent"])
print("Game key:", result["game_key"])
print("Filter:", result["metadata_filter"])

print("\nAnswer:")
print(result["answer"])

print("\nRetrieved sources:")
for i, doc in enumerate(result["retrieved_docs"], start=1):
    print(
        i,
        Path(doc.metadata.get("source", "")).name,
        doc.metadata.get("game_key"),
        doc.metadata.get("section"),
    )

Intent: review
Game key: cyberpunk_2077
Filter: {'$and': [{'section': {'$eq': 'review'}}, {'game_key': {'$eq': 'cyberpunk_2077'}}]}

Answer:
최근 리뷰는 대체로 긍정적입니다.

주요 반응:
- 스토리·캐릭터·연출을 높이 평가 ("Such a beautiful and well-crafted story.", "Great story.")
- 몰입감·재미를 강조 ("Super fun and immersive game. Love getting lost in it.", "Its fun")
- 모딩 지원과 추가 실험을 추천하는 의견도 있음 ("Amazing modding support...")
- 플레이시간이 긴 리뷰들이 많아 높은 몰입/재방문도가 확인됨 (예: 197.4h, 302.4h, 702.6h 등)
- 간단한 찬사형 코멘트 다수 ("Great", "greatest game oat", "love this game")

전반적으로 최근 리뷰는 긍정적이고 만족도가 높은 편입니다.

Retrieved sources:
1 cyberpunk_2077.md cyberpunk_2077 review
2 cyberpunk_2077.md cyberpunk_2077 review
3 cyberpunk_2077.md cyberpunk_2077 review
4 cyberpunk_2077.md cyberpunk_2077 review


In [28]:
eval_questions = [
    "Hollow Knight는 어떤 플레이 스타일의 게임인가요?",
    "Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?",
    "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?",
    "Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?",
    "Baldur's Gate 3는 어떤 RPG인가요?",
    "Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?",
    "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?",
    "No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?",
    "Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?",
    "Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?",
]

len(eval_questions)

10

In [29]:
import pandas as pd

records = []

for q in eval_questions:
    print("=" * 80)
    print("Question:", q)

    result = run_rag(q)

    retrieved_info = []
    for doc in result["retrieved_docs"]:
        retrieved_info.append(
            {
                "source": Path(doc.metadata.get("source", "")).name,
                "section": doc.metadata.get("section", "unknown"),
                "preview": doc.page_content[:300].replace("\n", " "),
            }
        )

    records.append(
        {
            "question": q,
            "intent": result["intent"],
            "answer": result["answer"],
            "retrieved_sources": json.dumps(retrieved_info, ensure_ascii=False),
        }
    )

    print("Intent:", result["intent"])
    print("Answer:", result["answer"][:500])
    print()

eval_df = pd.DataFrame(records)
eval_df

Question: Hollow Knight는 어떤 플레이 스타일의 게임인가요?
Intent: general
Answer: Hollow Knight는 고전 스타일의 2D 횡스크롤 액션 어드벤처입니다. 타이트하게 튜닝된 조작으로 회피·대시·베기 중심의 전투를 즐기며, 넓고 상호 연결된 세계를 자유롭게 탐험하고 길을 개척해 나가는 플레이가 핵심입니다. 손으로 그린 고딕 풍의 세계와 기괴한 벌레들로 이루어진 지역들을 발견하고 NPC와 교류하거나 비밀을 풀어가는 요소도 있습니다. 싱글플레이 기반이며, 게임을 완료하면 더 높은 난이도의 Steel Soul Mode가 잠금 해제되는 등 도전 요소가 강한 편입니다.

Question: Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?
Intent: general
Answer: 짧게 요약하면 다음과 같습니다.

- 분위기: 음울하고 고딕한 정서가 강한 수공(手工)적 연출. 귀엽지만 오싹한 캐릭터들과 기괴하고 때로는 무서운 생물들이 등장하며, Christopher Larkin의 잔잔하면서도 애잔한 음악이 멸망한 문명의 장엄함과 슬픔을 강조합니다.  
- 비주얼·연출: 전통 2D 손그림 애니메이션으로 세밀하게 표현된 지역들, 페인티드 풍경과 화려한 패럴랙스(입체감)가 옆보기 화면에 깊이감을 부여합니다.  
- 월드 구성: 넓고 상호 연결된(Hub-like) 세계—뒤얽힌 동굴, 고대 도시, 버려진 고속로, 무성한 야생지, 폐허 등 다양한 구역이 연결되어 있고 각 구역이 독특하고 낯설게 설계되어 있습니다. 플레이어가 길을 선택해 스스로 길을 개척하는 탐험 중심 구조이며, 지도를 사고 업그레이드해 여정을 기록할 수 있는 풍부한 맵핑 시스템도 제공됩니다.  
- 추가 요소: Dream Nail로 등장인물·적의 내면(다른 면모)을 파고들 수 있어 세계관의 미스터리를 더

Question: Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?
Intent: general
Answer: 간단히 말하면 반

,question,intent,answer,retrieved_sources
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,general,Hollow Knight는 고전 스타일의 2D 횡스크롤 액션 어드벤처입니다. 타이트...,"[{""source"": ""hollow_knight.md"", ""section"": ""ab..."
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,general,짧게 요약하면 다음과 같습니다.\n\n- 분위기: 음울하고 고딕한 정서가 강한 수공...,"[{""source"": ""hollow_knight.md"", ""section"": ""ab..."
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,general,간단히 말하면 반복되는 핵심 루프는 다음과 같습니다.\n\n1. 퀘스트를 받아 다양...,"[{""source"": ""monster_hunter_world.md"", ""sectio..."
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,general,Monster Hunter: World는 혼자 플레이할 수도 있고 최대 3명의 다른...,"[{""source"": ""monster_hunter_world.md"", ""sectio..."
4,Baldur's Gate 3는 어떤 RPG인가요?,general,Baldur’s Gate 3는 던전 앤 드래곤즈 세계관을 바탕으로 한 스토리 중심의...,"[{""source"": ""baldurs_gate_3.md"", ""section"": ""s..."
5,Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?,general,Baldur's Gate 3에서 선택은 서사의 핵심입니다. 플레이어의 행동과 선택이...,"[{""source"": ""baldurs_gate_3.md"", ""section"": ""s..."
6,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,news,간단히 요약하면 최근(2026년 4월) No Man's Sky 업데이트는 'Xeno...,"[{""source"": ""no_mans_sky.md"", ""section"": ""news..."
7,No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?,news,최근 No Man's Sky 업데이트 방향은 'Xeno Arena'(2026년 4월...,"[{""source"": ""no_mans_sky.md"", ""section"": ""news..."
8,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,review,"최근 스팀 리뷰는 대체로 긍정적입니다. 리뷰들에서 “재미있다/재밌음”, “몰입감 있...","[{""source"": ""cyberpunk_2077.md"", ""section"": ""r..."
9,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,general,문서에서 확인되는 주요 특징은 다음과 같습니다.\n\n- 장르·무대: 오픈월드 액션...,"[{""source"": ""cyberpunk_2077.md"", ""section"": ""a..."


In [30]:
EVAL_DIR = PROJECT_ROOT / "data" / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

output_csv = EVAL_DIR / "week3_baseline_results.csv"
eval_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("saved:", output_csv)

saved: C:\Users\asguug\Documents\rag-agent\data\eval\week3_baseline_results.csv


In [31]:
ragas_questions = [
    "Hollow Knight는 어떤 플레이 스타일의 게임인가요?",
    "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?",
    "Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?",
]

ragas_references = [
    "Hollow Knight는 2D 사이드스크롤 액션 어드벤처 게임으로, 거대한 연결형 세계를 탐험하고 적과 전투하며 고대 왕국의 비밀을 밝혀가는 플레이 스타일을 가진다.",
    "No Man's Sky는 지속적인 업데이트를 통해 새로운 콘텐츠와 시스템을 추가해 왔으며, 최근 뉴스에서는 새로운 업데이트와 게임 확장 방향이 언급된다.",
    "Cyberpunk 2077의 최근 Steam 리뷰는 대체로 긍정적인 반응이 많으며, 스토리, 캐릭터, 몰입감, 모딩 지원 등에 대한 호평이 확인된다.",
]

len(ragas_questions), len(ragas_references)

(3, 3)

In [32]:
ragas_records = []

for_q = list(zip(ragas_questions, ragas_references))

for question, reference in for_q:
    result = run_rag(question)

    contexts = [
        doc.page_content
        for doc in result["retrieved_docs"]
    ]

    ragas_records.append(
        {
            "question": question,
            "answer": result["answer"],
            "contexts": contexts,
            "ground_truth": reference,
            # 일부 RAGAS 버전 호환용 컬럼
            "user_input": question,
            "response": result["answer"],
            "retrieved_contexts": contexts,
            "reference": reference,
        }
    )

ragas_input_df = pd.DataFrame(ragas_records)

ragas_input_df[["question", "answer", "ground_truth"]]

,question,answer,ground_truth
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,Hollow Knight는 클래식 스타일의 2D 사이드스크롤 액션 어드벤처입니다. ...,"Hollow Knight는 2D 사이드스크롤 액션 어드벤처 게임으로, 거대한 연결형..."
1,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,"최근 No Man's Sky의 대규모 업데이트는 ""Xeno Arena""(2026년 ...",No Man's Sky는 지속적인 업데이트를 통해 새로운 콘텐츠와 시스템을 추가해 ...
2,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,제공된 최근 스팀 리뷰들은 대체로 모두 긍정적입니다. 리뷰들에서 공통적으로 언급되는...,"Cyberpunk 2077의 최근 Steam 리뷰는 대체로 긍정적인 반응이 많으며,..."


In [33]:
from datasets import Dataset
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

C:\Users\asguug\AppData\Local\Temp\ipykernel_30604\1486504553.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykernel_30604\1486504553.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykernel_30604\1486504553.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykerne

In [35]:
ragas_dataset = Dataset.from_dict(
    {
        "question": ragas_input_df["question"].tolist(),
        "answer": ragas_input_df["answer"].tolist(),
        "contexts": ragas_input_df["contexts"].tolist(),
        "ground_truth": ragas_input_df["ground_truth"].tolist(),
    }
)

ragas_dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 3
})

In [42]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

ragas_eval_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

evaluator_llm = LangchainLLMWrapper(ragas_eval_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
]

# metric 객체에 평가용 LLM / embedding 명시 주입
for metric in metrics:
    if hasattr(metric, "llm"):
        metric.llm = evaluator_llm
    if hasattr(metric, "embeddings"):
        metric.embeddings = evaluator_embeddings

ragas_result = evaluate(
    ragas_dataset,
    metrics=metrics,
)

ragas_result

C:\Users\asguug\AppData\Local\Temp\ipykernel_30604\663037370.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykernel_30604\663037370.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykernel_30604\663037370.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykernel_3

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.9259, 'answer_relevancy': 0.4284, 'context_precision': 1.0000, 'context_recall': 1.0000}